In [12]:
import pandas as pd
import requests

API_KEY = "kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg"

In [13]:
state_df = pd.read_csv("state_production.csv")

state_df["month"] = pd.to_datetime(state_df["month"])

start_date = state_df["month"].min().to_period("M").start_time
end_date   = state_df["month"].max().to_period("M").end_time

print("State data starts:", start_date)
print("State data ends:", end_date)

State data starts: 2006-09-01 00:00:00
State data ends: 2026-06-30 23:59:59.999999


In [14]:
def fetch_eia_series(series_id, column_name):
    
    url = f"https://api.eia.gov/v2/seriesid/{series_id}"
    
    all_rows = []
    offset = 0
    batch_size = 5000

    while True:
        
        params = {
            "api_key": API_KEY,
            "length": batch_size,
            "offset": offset
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        rows = response.json()["response"]["data"]

        all_rows.extend(rows)

        if len(rows) < batch_size:
            break

        offset += batch_size

    df = pd.DataFrame(all_rows)

    df = df[["period", "value"]]

    df.columns = ["date", column_name]

    df["date"] = pd.to_datetime(df["date"])
    df[column_name] = pd.to_numeric(df[column_name], errors="coerce")

    return df

In [15]:
brent_daily = fetch_eia_series(
    "PET.RBRTE.D",
    "brent_price"
)

wti_daily = fetch_eia_series(
    "PET.RWTC.D",
    "wti_price"
)

print(brent_daily.shape)
print(wti_daily.shape)

(9968, 2)
(10240, 2)


In [16]:
brent_daily = brent_daily[
    (brent_daily["date"] >= start_date) &
    (brent_daily["date"] <= end_date)
]

wti_daily = wti_daily[
    (wti_daily["date"] >= start_date) &
    (wti_daily["date"] <= end_date)
]

In [17]:
daily_prices = pd.merge(
    brent_daily,
    wti_daily,
    on="date",
    how="outer"
)

daily_prices = daily_prices.sort_values("date")

daily_prices.head()

,date,brent_price,wti_price
0,2006-09-01,70.49,69.24
1,2006-09-04,68.62,NaN
2,2006-09-05,65.94,68.70
3,2006-09-06,65.41,67.75
4,2006-09-07,64.52,67.37


In [18]:
daily_prices["month"] = daily_prices["date"].dt.to_period("M")

monthly_prices = (
    daily_prices
    .groupby("month")[["brent_price", "wti_price"]]
    .mean()
    .reset_index()
)

monthly_prices.head()

,month,brent_price,wti_price
0,2006-09,61.958571,63.798000
1,2006-10,57.808182,58.893182
2,2006-11,58.761364,59.082500
3,2006-12,62.472105,61.959000
4,2007-01,53.681429,54.506190


In [19]:
monthly_prices.to_csv(
    "oil_prices.csv",
    index=False
)

## Oil Price Data

Daily crude-oil price data was fetched from the **U.S. Energy Information Administration (EIA) API** for two major benchmark prices:

| Feature | Meaning | Unit |
|---|---|---|
| `brent_price` | Brent crude-oil benchmark price | USD per barrel |
| `wti_price` | West Texas Intermediate crude-oil benchmark price | USD per barrel |

The original EIA series were available at **daily frequency**. Daily prices were converted into **monthly average prices** to match the monthly crude-production dataset.

The final oil-price dataset therefore contains:

`month | brent_price | wti_price`

where each row represents the average crude-oil benchmark prices for one month.